In [ ]:
import pandas as pd 
import os 
import csv  
import numpy as np
import matplotlib.pyplot as plt 
from scipy.stats import mannwhitneyu
from scipy.stats import kruskal
from itertools import combinations
from scipy.stats import  spearmanr
from statsmodels.stats.multitest import multipletests

#--------------------------------------
# change the path !! 
#--------------------------------------
df = pd.read_csv("/path/to/MELD-PostOp/model_performance/analysis.csv")
value_col = "DSC" 

In [ ]:
def iqr(series):
    return series.quantile(0.75) - series.quantile(0.25)

def add_sig_bar(x1, x2, y, h, text):
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, c='k')
    plt.text((x1+x2)/2, y+h+0.01, text, ha='center', va='center', fontsize=8)

def stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'n'

def pairwise_mwu(a, b, name_a, name_b):
        stat, p = mannwhitneyu(a, b, alternative="two-sided")
        print(f"{name_a} vs {name_b}: U = {stat:.3f}, p = {p:.4e}")
        return stat,p

global_mwu_pvals = []

def record_mwu_p(name, p):
    global global_mwu_pvals
    global_mwu_pvals.append({"Comparison": name, "p_value": p})


### Age 
paediatric vs adult (vs unknown)

In [85]:
df["age"] = df["age_at_surgery"].fillna(df["age_at_preop"])
paediatric_mask = (df["age"] < 18) 
adult_mask = (df["age"] >= 18) 
unknown_mask =df["age_at_surgery"].isna() & df["age_at_preop"].isna()

paediatric = df[paediatric_mask][value_col].dropna()
adult = df[adult_mask][value_col].dropna()
unknownage = df[unknown_mask][value_col].dropna()
data_age = [paediatric,adult]

stat, p = mannwhitneyu(paediatric, adult, alternative='greater')
total_n = len(paediatric) + len(adult)
print(f"\nMann–Whitney U statistic = {stat:.3f}, p-value = {p:.3e}")
record_mwu_p("paed-adult",p)
results = []
results.append({
    "Group": "Paediatric",
    "Count": len(paediatric),
    "Percentage": 100 * len(paediatric) / total_n,
    "Total Count": total_n,
    "minimum": paediatric.min(),
    "maximum": paediatric.max(),
    "Mean ± SD": f"{paediatric.mean():.3f} ± {paediatric.std():.3f}",
    "Median (IQR)": f"{paediatric.median():.3f} ({iqr(paediatric):.3f})"
})

results.append({
    "Group": "Adult",
    "Count": len(adult),
    "Percentage": 100 * len(adult) / total_n,
    "Total Count": total_n,
    "minimum": adult.min(),
    "maximum": adult.max(),
    "Mean ± SD": f"{adult.mean():.3f} ± {adult.std():.3f}",
    "Median (IQR)": f"{adult.median():.3f} ({iqr(adult):.3f})"
})


summary_df = pd.DataFrame(results).sort_values(by="Group").reset_index(drop=True)
display(summary_df)


Mann–Whitney U statistic = 0.000, p-value = 9.964e-01


,Group,Count,Percentage,Total Count,minimum,maximum,Mean ± SD,Median (IQR)
0,Adult,6,60.0,10,0.88,0.942101,0.914 ± 0.020,0.915 (0.005)
1,Paediatric,4,40.0,10,0.00,0.761858,0.381 ± 0.440,0.381 (0.762)


### Sex 
male vs female

In [86]:
df['sex_group'] = df['sex'].replace({0: 'Male', 1: 'Female', np.nan: 'Unknown'})
male = df[df['sex_group'] == 'Male'][value_col].dropna()
female = df[df['sex_group'] == 'Female'][value_col].dropna()

data_sex = [male,female]
stat,p = mannwhitneyu(male,female,alternative='greater')

record_mwu_p("male-female",p)

print(f"Mann-Whitney U statistic: {stat}, p-value: {p}")


total_n = len(male) + len(female) 


results = [] 
results.append({
    "Group": "Male",
    "Count": len(male),
    "Percentage": 100 * len(male) / total_n,
    "Total Count": total_n,
    "minimum": male.min(),
    "maximum": male.max(),
    "Mean ± SD": f"{male.mean():.3f} ± {male.std():.3f}",
    "Median (IQR)": f"{male.median():.3f} ({iqr(male):.3f})"
})

results.append({
    "Group": "Female",
    "Count": len(female),
    "Percentage": 100 * len(female) / total_n,
    "Total Count": total_n,
    "minimum": female.min(),
    "maximum": female.max(),
    "Mean ± SD": f"{female.mean():.3f} ± {female.std():.3f}",
    "Median (IQR)": f"{female.median():.3f} ({iqr(female):.3f})"
})



summary_df = pd.DataFrame(results).sort_values(by=["Group"]).reset_index(drop=True)
display(summary_df)

Mann-Whitney U statistic: 8.0, p-value: 0.833539195967217


,Group,Count,Percentage,Total Count,minimum,maximum,Mean ± SD,Median (IQR)
0,Female,6,60.0,10,0.000000,0.942101,0.615 ± 0.477,0.915 (0.690)
1,Male,4,40.0,10,0.761858,0.913275,0.829 ± 0.079,0.821 (0.126)


### Resection side 
LH vs RH

In [87]:
LH= df[df["resection_side"] == "L"][value_col].dropna()
RH = df[df["resection_side"] == "R"][value_col].dropna()
data_side = [LH,RH]

stat,p = mannwhitneyu(LH,RH,alternative='greater')
total_n = len(LH) + len(RH)
print(f"Mann-Whitney U statistic: {stat}, p-value: {p}")
record_mwu_p("lh-rh",p)
results = [] 
results.append({
    "Group": "L",
    "Count": len(LH),
    "Percentage": 100 * len(LH) / total_n,
    "Total Count": total_n,
    "minimum": LH.min(),
    "maximum": LH.max(),
    "Mean ± SD": f"{LH.mean():.3f} ± {LH.std():.3f}",
    "Median (IQR)": f"{LH.median():.3f} ({iqr(LH):.3f})"
})

results.append({ 
    "Group": "R",
    "Count": len(RH),
    "Percentage": 100 * len(RH) / total_n,
    "Total Count": total_n,
    "minimum": RH.min(),
    "maximum": RH.max(),
    "Mean ± SD": f"{RH.mean():.3f} ± {RH.std():.3f}",
    "Median (IQR)": f"{RH.median():.3f} ({iqr(RH):.3f})"
})

summary_df = pd.DataFrame(results).sort_values(by="Group").reset_index(drop=True)
display(summary_df)


Mann-Whitney U statistic: 11.5, p-value: 0.6240851829770754


,Group,Count,Percentage,Total Count,minimum,maximum,Mean ± SD,Median (IQR)
0,L,5,50.0,10,0.000000,0.942101,0.554 ± 0.506,0.913 (0.915)
1,R,5,50.0,10,0.761858,0.920000,0.848 ± 0.080,0.880 (0.153)


### Resection lobe 
Frontal vs Temporal vs Occipital vs Parietal vs Limbic vs Insular vs Other 

In [88]:
df['lobe_group'] = df['resection_lobe'].fillna('other')

label_lobe = ['frontal','temporal','parietal','occipital','limbic','insular','other']
group_lobe = {g: df[df['lobe_group'] == g][value_col].dropna() for g in label_lobe}
data_lobe = [group_lobe[g] for g in label_lobe]
total_n = sum(len(v) for v in data_lobe)

stat, p = kruskal(*data_lobe)
print(f"Kruskal-Wallis H = {stat:.3f}, p = {p:.3e}")

if p < 0.05:
    print("\n✅ Significant overall difference — running pairwise Mann–Whitney tests...\n")
    pairwise_p = {}
    for g1, g2 in combinations(label_lobe, 2):
        d1, d2 = group_lobe[g1], group_lobe[g2]
        if len(d1) > 0 and len(d2) > 0:
            _, p_val = pairwise_mwu(d1, d2, g1, g2)
            pairwise_p[f"{g1}–{g2}"] = p_val
            record_mwu_p(f"lobe_{g1}_vs_{g2}", p_val) 
    
    
else:
    print("❌ No statistically significant difference among pathology groups.")


results = []
for g in label_lobe:
    vals = group_lobe[g]
    if len(vals) > 0:
        results.append({
            "Group": g,
            "Count": len(vals),
            "Percentage": 100 * len(vals) / total_n,
            "Total Count": total_n,
            "minimum": vals.min(),
            "maximum": vals.max(),
            "Mean ± SD": f"{vals.mean():.3f} ± {vals.std():.3f}",
            "Median (IQR)": f"{vals.median():.3f} ({iqr(vals):.3f})",
            "(IQR)": f"{vals.quantile(0.75):.3f} - {vals.quantile(0.25):.3f})"
        })
    else:
        results.append({
            "Group": g,
            "Count": 0,
            "Percentage": 0,
            "Total Count": total_n,
            "minimum": np.nan,
            "maximum": np.nan,
            "Mean ± SD": "NA",
            "Median (IQR)": "NA"
        })

summary_df = pd.DataFrame(results)
display(summary_df)


Kruskal-Wallis H = 8.579, p = 1.987e-01
❌ No statistically significant difference among pathology groups.


,Group,Count,Percentage,Total Count,minimum,maximum,Mean ± SD,Median (IQR),(IQR)
0,frontal,2,20.0,10,0.915232,0.920000,0.918 ± 0.003,0.918 (0.002),0.919 - 0.916)
1,temporal,3,30.0,10,0.000000,0.761858,0.254 ± 0.440,0.000 (0.381),0.381 - 0.000)
2,parietal,1,10.0,10,0.913275,0.913275,0.913 ± nan,0.913 (0.000),0.913 - 0.913)
3,occipital,1,10.0,10,0.880000,0.880000,0.880 ± nan,0.880 (0.000),0.880 - 0.880)
4,limbic,1,10.0,10,0.942101,0.942101,0.942 ± nan,0.942 (0.000),0.942 - 0.942)
5,insular,1,10.0,10,0.761858,0.761858,0.762 ± nan,0.762 (0.000),0.762 - 0.762)
6,other,1,10.0,10,0.915232,0.915232,0.915 ± nan,0.915 (0.000),0.915 - 0.915)


### Pathology 
MCD/FCD vs LEAT vs Cavernoma vs HS vs Dual pathology vs other (vs unknown)

In [89]:
pathology_map = {
    1: 'FCD 1', 2: 'FCD 2A', 3: 'FCD 2B', 4: 'FCD 3A', 5: 'FCD 3B',
    6: 'FCD 3C', 7: 'FCD 3D', 8: 'FCD 2 other', 9: 'FCD other', 10: 'HS',
    11: 'Hippocampal gliosis', 12: 'Cortical gliosis', 13: 'DNET', 14: 'Ganglioglioma',
    15: 'Other LEAT', 16: 'Polymicrogyria', 17: 'PNH', 18: 'Cavernoma', 19: 'Non-specific',
    21: 'Normal', 22: 'HH', 23: 'Other', 25: 'MOGHE', 26: 'MCD', 27: 'LEAT xdef',
    28: 'Astrocytoma', 29: 'PLNTY', 30: 'mMCD', 31: 'Glioblastoma/highgradetumour'
}

group_map = {
    'FCD 1': 'MCD/FCD', 'FCD 2A': 'MCD/FCD', 'FCD 2B': 'MCD/FCD',
    'FCD 3B': 'MCD/FCD', 'FCD 2 other': 'MCD/FCD', 'FCD other': 'MCD/FCD',
    'MOGHE': 'MCD/FCD', 'MCD': 'MCD/FCD', 'mMCD': 'MCD/FCD', 'Polymicrogyria': 'MCD/FCD',
    'Cavernoma': 'Cavernoma',
    'Cortical gliosis': 'Other', 'Normal': 'Other', 'HH': 'Other', 'PNH': 'Other', 'Other': 'Other',
    'HS': 'HS', 'Hippocampal gliosis': 'HS',
    'DNET': 'LEAT', 'Ganglioglioma': 'LEAT', 'Other LEAT': 'LEAT', 
    'LEAT xdef': 'LEAT', 'Astrocytoma': 'LEAT', 'PLNTY': 'LEAT', 
    'Glioblastoma/highgradetumour': 'LEAT',
    'Dual Pathology': 'Dual',
    'unknown': 'Unknown'
}


missing_vals = ['555', '', ' ']
df['pathology'] = df['pathology'].replace(missing_vals, pd.NA)
df['pathology_other'] = df['pathology_other'].replace(missing_vals, pd.NA)
df['pathology'] = pd.to_numeric(df['pathology'], errors='coerce')
df['pathology_other'] = pd.to_numeric(df['pathology_other'], errors='coerce')

# --- Identify dual vs single pathology ---
df['is_dual'] = df['pathology_other'].notna()

# --- Map primary pathology to grouped category ---
df['pathology_label'] = df['pathology'].map(pathology_map)
df['pathology_label'] = df['pathology_label'].fillna('unknown')
df['path_group'] = df['pathology_label'].map(group_map).fillna('Unknown')

# --- Override with 'Dual' if classification_notes is filled ---
df.loc[df['is_dual'], 'path_group'] = 'Dual'

# --- Define groups ---
label_path = ['MCD/FCD', 'HS', 'LEAT', 'Dual', 'Cavernoma', 'Other', 'Unknown']
group_data = {g: df[df['path_group'] == g][value_col].dropna() for g in label_path}
data_path = [group_data[g] for g in label_path]
total_n = sum(len(v) for v in data_path)

stat, p = kruskal(*data_path)
print(f"Kruskal-Wallis H = {stat:.3f}, p = {p:.3e}")

if p < 0.05:
    print("\n✅ Significant overall difference — running pairwise Mann–Whitney tests...\n")
    pairwise_p = {}
    for g1, g2 in combinations(label_path, 2):
        d1, d2 = group_data[g1], group_data[g2]
        if len(d1) > 0 and len(d2) > 0:
            _, p_val = pairwise_mwu(d1, d2, g1, g2)
            pairwise_p[f"{g1}–{g2}"] = p_val
            record_mwu_p(f"path_{g1}_vs_{g2}", p_val) 

else:
    print("❌ No statistically significant difference among pathology groups.")


results = []
for g in label_path:
    vals = group_data[g]
    if len(vals) > 0:
        results.append({
            "Group": g,
            "Count": len(vals),
            "Percentage": 100 * len(vals) / total_n,
            "Total Count": total_n,
            "minimum": vals.min(),
            "maximum": vals.max(),
            "Mean ± SD": f"{vals.mean():.3f} ± {vals.std():.3f}",
            "Median (IQR)": f"{vals.median():.3f} ({iqr(vals):.3f})"
        })
    else:
        results.append({
            "Group": g,
            "Count": 0,
            "Percentage": 0,
            "Total Count": total_n,
            "minimum": np.nan,
            "maximum": np.nan,
            "Mean ± SD": "NA",
            "Median (IQR)": "NA"
        })

summary_df = pd.DataFrame(results)
display(summary_df)

Kruskal-Wallis H = nan, p = nan
❌ No statistically significant difference among pathology groups.


/tmp/ipykernel_785779/183980515.py:48: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  stat, p = kruskal(*data_path)


,Group,Count,Percentage,Total Count,minimum,maximum,Mean ± SD,Median (IQR)
0,MCD/FCD,4,40.0,10,0.000000,0.920000,0.450 ± 0.520,0.440 (0.890)
1,HS,2,20.0,10,0.915232,0.915232,0.915 ± 0.000,0.915 (0.000)
2,LEAT,2,20.0,10,0.761858,0.913275,0.838 ± 0.107,0.838 (0.076)
3,Dual,1,10.0,10,0.761858,0.761858,0.762 ± nan,0.762 (0.000)
4,Cavernoma,0,0.0,10,NaN,NaN,NA,NA
5,Other,0,0.0,10,NaN,NaN,NA,NA
6,Unknown,1,10.0,10,0.942101,0.942101,0.942 ± nan,0.942 (0.000)


### Image isotropy 
isotropic vs anisotropic 

In [90]:

iso = df[df["image_isotropy"] == "isotropic"][value_col].dropna()
aniso = df[df["image_isotropy"] == "anisotropic"][value_col].dropna()
data_iso = [iso,aniso]

stat, p = mannwhitneyu(iso, aniso, alternative='greater')
total_n = len(iso) + len(aniso)
print(f"\nMann–Whitney U statistic = {stat:.3f}, p-value = {p:.3e}")
record_mwu_p("iso-aniso",p)
results = []
results.append({
    "Group": "isotropic",
    "Count": len(iso),
    "Percentage": 100 * len(iso) / total_n,
    "Total Count": total_n,
    "minimum": iso.min(),
    "maximum": iso.max(),
    "Mean ± SD": f"{iso.mean():.3f} ± {iso.std():.3f}",
    "Median (IQR)": f"{iso.median():.3f} ({iqr(iso):.3f})"
})

results.append({
    "Group": "anisotropic",
    "Count": len(aniso),
    "Percentage": 100 * len(aniso) / total_n,
    "Total Count": total_n,
    "minimum": aniso.min(),
    "maximum": aniso.max(),
    "Mean ± SD": f"{aniso.mean():.3f} ± {aniso.std():.3f}",
    "Median (IQR)": f"{aniso.median():.3f} ({iqr(aniso):.3f})"
})

summary_df = pd.DataFrame(results).sort_values(by="Group").reset_index(drop=True)
display(summary_df)



Mann–Whitney U statistic = 4.000, p-value = 8.822e-01


,Group,Count,Percentage,Total Count,minimum,maximum,Mean ± SD,Median (IQR)
0,anisotropic,2,20.0,10,0.915232,0.915232,0.915 ± 0.000,0.915 (0.000)
1,isotropic,8,80.0,10,0.000000,0.942101,0.647 ± 0.405,0.821 (0.344)


### Field strength 
3T vs 1.5T 

In [91]:
df["field_strength"] = df["field_strength"].replace(["", " ", np.nan], "Unknown")
threeT = df[df["field_strength"] == "3T"][value_col].dropna()
oneT   = df[df["field_strength"] == "1.5T"][value_col].dropna()

data_t = [threeT,oneT]

stat,p = mannwhitneyu(threeT, oneT, alternative='greater')
print(f"mann whitney u test: H = {stat:.3f}, p = {p:.4e}")

record_mwu_p("3t-1.5t",p)

results = [] 
results.append({
    "Group": "3T",
    "Count": len(threeT),
    "Percentage": 100 * len(threeT) / total_n,
    "Total Count": total_n,
    "minimum": threeT.min(),
    "maximum": threeT.max(),
    "Mean ± SD": f"{threeT.mean():.3f} ± {threeT.std():.3f}",
    "Median (IQR)": f"{threeT.median():.3f} ({iqr(threeT):.3f})"
})

results.append({
    "Group": "1.5T",
    "Count": len(oneT),
    "Percentage": 100 * len(oneT) / total_n,
    "Total Count": total_n,
    "minimum": oneT.min(),
    "maximum": oneT.max(),
    "Mean ± SD": f"{oneT.mean():.3f} ± {oneT.std():.3f}",
    "Median (IQR)": f"{oneT.median():.3f} ({iqr(oneT):.3f})"
})

summary_df = pd.DataFrame(results).sort_values(by=["Group"]).reset_index(drop=True)
display(summary_df)

mann whitney u test: H = 3.000, p = 9.6713e-01


,Group,Count,Percentage,Total Count,minimum,maximum,Mean ± SD,Median (IQR)
0,1.5T,3,30.0,10,0.88,0.942101,0.914 ± 0.031,0.920 (0.031)
1,3T,7,70.0,10,0.00,0.915232,0.610 ± 0.422,0.762 (0.533)


### ground truth volume

In [92]:
df = df.dropna(subset=["manual_volume", "DSC"])

df["volume"] = df["manual_volume"] / 1000

sp_corr, sp_p = spearmanr(df["volume"], df["DSC"])

print("\nPearson correlation:")
print(f"r = {sp_corr:.3f}, p = {sp_p:.6f}")
record_mwu_p("volume", sp_p) 



Pearson correlation:
r = 0.745, p = 0.013491


### p correction and multiple tests 

In [ ]:
pvals_df = pd.DataFrame(global_mwu_pvals)
reject_bonf, pvals_bonf, _, _ = multipletests(pvals_df["p_value"], method='bonferroni')
pvals_df["Bonferroni_p"] = pvals_bonf
pvals_df["Reject_Bonferroni"] = reject_bonf
display(pvals_df)

#--------------------------------------
# change the path !! 
#--------------------------------------
csv_results_out = "/path/to/MELD-PostOp/model_performance"
out_csv=os.path.join(csv_results_out,"performance_analysis_results.csv")
pvals_df.to_csv(out_csv,index=False)

,Comparison,p_value,Bonferroni_p,Reject_Bonferroni
0,paed-adult,0.996423,1.000000,False
1,male-female,0.833539,1.000000,False
2,lh-rh,0.624085,1.000000,False
3,iso-aniso,0.882160,1.000000,False
4,3t-1.5t,0.967129,1.000000,False
5,volume,0.013491,0.080947,False


OSError: Cannot save file into a non-existent directory: '/path/to/MELD-PostOp/model_performance'